In [ ]:
import pandas as pd
import numpy as np
from feature_engineering import FeatureEngineering

In [2]:
df = pd.read_csv("clinic_visits.csv")
df

,Patient,Date,Day,Visit Time
0,Sneha K,2026-07-01,Wed,8:13 AM
1,Vijay N,2026-07-01,Wed,9:01 AM
2,Sneha V,2026-07-01,Wed,9:01 AM
3,Sanjay Nair,2026-07-01,Wed,9:29 AM
4,Lakshmi Babu,2026-07-01,Wed,9:39 AM
...,...,...,...,...
1332,Suresh M,2026-08-29,Sat,5:38 PM
1333,Lakshmi Nair,2026-08-29,Sat,5:44 PM
1334,Kiran Nathan,2026-08-29,Sat,6:08 PM
1335,Vikram P,2026-08-29,Sat,7:05 PM


In [3]:
df['visit_dt'] = pd.to_datetime(
    df['Date'] + ' ' + df['Visit Time'],
    format='%Y-%m-%d %I:%M %p'
)

df['hour_slot'] = df['visit_dt'].dt.floor('h')

hourly_df = (df.groupby('hour_slot').size().reset_index(name='visit_count'))

In [4]:
hourly_df['Date'] = (
    hourly_df['hour_slot']
    .dt.strftime('%Y-%m-%d')
)

hourly_df['Visit Time'] = (
    hourly_df['hour_slot']
    .dt.strftime('%I:%M %p')
)

In [5]:
X = hourly_df[['Date','Visit Time']]
y = hourly_df['visit_count']

In [6]:
hourly_df = (hourly_df.sort_values('hour_slot').reset_index(drop=True))

In [7]:
hourly_df

,hour_slot,visit_count,Date,Visit Time
0,2026-07-01 08:00:00,1,2026-07-01,08:00 AM
1,2026-07-01 09:00:00,4,2026-07-01,09:00 AM
2,2026-07-01 10:00:00,4,2026-07-01,10:00 AM
3,2026-07-01 17:00:00,4,2026-07-01,05:00 PM
4,2026-07-01 18:00:00,4,2026-07-01,06:00 PM
...,...,...,...,...
413,2026-08-29 10:00:00,6,2026-08-29,10:00 AM
414,2026-08-29 17:00:00,3,2026-08-29,05:00 PM
415,2026-08-29 18:00:00,1,2026-08-29,06:00 PM
416,2026-08-29 19:00:00,1,2026-08-29,07:00 PM


In [10]:
split_idx = int(len(hourly_df) * 0.8)
train_df = (hourly_df.iloc[:split_idx])
test_df = (hourly_df.iloc[split_idx:])

In [11]:
train_df

,hour_slot,visit_count,Date,Visit Time
0,2026-07-01 08:00:00,1,2026-07-01,08:00 AM
1,2026-07-01 09:00:00,4,2026-07-01,09:00 AM
2,2026-07-01 10:00:00,4,2026-07-01,10:00 AM
3,2026-07-01 17:00:00,4,2026-07-01,05:00 PM
4,2026-07-01 18:00:00,4,2026-07-01,06:00 PM
...,...,...,...,...
329,2026-08-17 09:00:00,5,2026-08-17,09:00 AM
330,2026-08-17 10:00:00,4,2026-08-17,10:00 AM
331,2026-08-17 17:00:00,4,2026-08-17,05:00 PM
332,2026-08-17 18:00:00,3,2026-08-17,06:00 PM


In [12]:
X_train = train_df[['Date','Visit Time']]

y_train = train_df['visit_count']

X_test = test_df[['Date','Visit Time']]

y_test = test_df['visit_count']

In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

In [14]:
preprocessor = (ColumnTransformer(transformers=[
            ('session',OneHotEncoder(handle_unknown='ignore'),['session'])],
        remainder='passthrough'))
        

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

In [16]:
pipe_rf = Pipeline([
    ('FeatureEngineering', FeatureEngineering()),
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ))
])

In [17]:
pipe_rf.fit(X_train, y_train)
y_pred = pipe_rf.predict(X_test)
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

In [18]:
print(r2)
print(mse)

0.6147600394410349
0.8353417512120418


In [325]:
import xgboost
from xgboost import XGBRegressor
pipe_xgb = Pipeline([
    ('FeatureEngineering', FeatureEngineering()),
    ('preprocessor', preprocessor),
    ('model', XGBRegressor(
        n_estimators=100,
        random_state=42
    ))
])
pipe_xgb.fit(X_train, y_train)
y_pred = pipe_xgb.predict(X_test)
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(r2)
print(mse)

0.6189101934432983
0.8263427019119263


In [326]:
from catboost import CatBoostRegressor
pipe_cat = Pipeline([
    ('FeatureEngineering', FeatureEngineering()),
    ('preprocessor', preprocessor),
    ('model', CatBoostRegressor())
])
pipe_cat.fit(X_train, y_train)
y_pred = pipe_cat.predict(X_test)
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(r2)
print(mse)

Learning rate set to 0.03443
0:	learn: 1.3898483	total: 2.23ms	remaining: 2.23s
1:	learn: 1.3587742	total: 4.04ms	remaining: 2.02s
2:	learn: 1.3289140	total: 5.91ms	remaining: 1.96s
3:	learn: 1.3007159	total: 8.3ms	remaining: 2.06s
4:	learn: 1.2776694	total: 9.98ms	remaining: 1.99s
5:	learn: 1.2536981	total: 12ms	remaining: 1.99s
6:	learn: 1.2278788	total: 13.5ms	remaining: 1.92s
7:	learn: 1.2085638	total: 14.8ms	remaining: 1.84s
8:	learn: 1.1852394	total: 16.7ms	remaining: 1.84s
9:	learn: 1.1624029	total: 18.4ms	remaining: 1.82s
10:	learn: 1.1407182	total: 20.2ms	remaining: 1.82s
11:	learn: 1.1197767	total: 21.8ms	remaining: 1.79s
12:	learn: 1.1042046	total: 23.3ms	remaining: 1.77s
13:	learn: 1.0896237	total: 24.5ms	remaining: 1.73s
14:	learn: 1.0705689	total: 26.1ms	remaining: 1.71s
15:	learn: 1.0524109	total: 27.7ms	remaining: 1.71s
16:	learn: 1.0354331	total: 30.1ms	remaining: 1.74s
17:	learn: 1.0194202	total: 32.4ms	remaining: 1.77s
18:	learn: 1.0078329	total: 33.8ms	remaining: 1.

In [327]:
from sklearn.linear_model import LinearRegression
pipe_lr = Pipeline([
    ('FeatureEngineering', FeatureEngineering()),
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])
pipe_lr.fit(X_train, y_train)
y_pred = pipe_lr.predict(X_test)
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(r2)
print(mse)


0.12324352072192568
1.9011301208835794


In [19]:
from sklearn.tree import DecisionTreeRegressor
pipe_dt = Pipeline([
    ('FeatureEngineering', FeatureEngineering()),
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(    
        random_state=42
    ))
])
pipe_dt.fit(X_train, y_train)
y_pred = pipe_dt.predict(X_test)
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(r2)
print(mse)
print(mae)

0.6187114845938375
0.8267735665694849
0.7137188208616779


In [330]:
import pickle

with open("clinic_model.pkl", "wb") as f:
    pickle.dump(pipe_dt, f)